# Fine-Tuning Large Language Models with Transformers: A Beginner's Guide

## Table of Contents
1. [Introduction to Fine-Tuning](#introduction)
2. [Understanding Pre-trained Models](#pretrained-models)
3. [Types of Fine-Tuning](#types-of-finetuning)
4. [Setting up the Environment](#setup)
5. [Data Preparation](#data-prep)
6. [Fine-Tuning Implementation](#implementation)
7. [Evaluation and Testing](#evaluation)
8. [Advanced Techniques](#advanced)
9. [Best Practices](#best-practices)
10. [Troubleshooting](#troubleshooting)

---

## 1. Introduction to Fine-Tuning {#introduction}

**Fine-tuning** is a transfer learning technique where we take a pre-trained model and adapt it to perform well on a specific task or domain. Instead of training a model from scratch (which requires massive computational resources and data), we leverage the knowledge already learned by large models like GPT, BERT, or T5.

### Why Fine-Tune?
- **Efficiency**: Much faster than training from scratch
- **Performance**: Often achieves better results than training from scratch
- **Resource-friendly**: Requires less computational power and data
- **Specialization**: Adapts general models to specific domains or tasks

### Key Concepts:
- **Transfer Learning**: Using knowledge from one task to improve performance on another
- **Domain Adaptation**: Adapting a model to work well in a specific field (medical, legal, etc.)
- **Task-specific Fine-tuning**: Training for specific tasks (classification, generation, etc.)

## 2. Understanding Pre-trained Models {#pretrained-models}

### What are Pre-trained Models?
Pre-trained models are neural networks that have already been trained on large datasets. They've learned general language patterns, syntax, semantics, and world knowledge.

### Popular Pre-trained Models:

| Model Family | Best For | Examples |
|--------------|----------|---------|
| **BERT** | Understanding tasks (classification, NER) | `bert-base-uncased`, `roberta-large` |
| **GPT** | Text generation | `gpt2`, `gpt-3.5-turbo` |
| **T5** | Text-to-text tasks | `t5-small`, `t5-base` |
| **DistilBERT** | Lightweight understanding | `distilbert-base-uncased` |
| **BART** | Sequence-to-sequence tasks | `facebook/bart-large` |

### Model Architecture Components:
- **Tokenizer**: Converts text to tokens (numbers)
- **Embeddings**: Convert tokens to vectors
- **Transformer Layers**: Process the information
- **Head**: Task-specific output layer

## 3. Types of Fine-Tuning {#types-of-finetuning}

### 3.1 Full Fine-Tuning
- **What**: Update all model parameters
- **When**: When you have sufficient data and computational resources
- **Pros**: Maximum adaptation to your task
- **Cons**: Computationally expensive, risk of overfitting

### 3.2 Parameter-Efficient Fine-Tuning (PEFT)

#### LoRA (Low-Rank Adaptation)
- **What**: Add small trainable matrices to existing layers
- **Benefits**: Reduces trainable parameters by 90%+
- **Best for**: Large models where full fine-tuning is impractical

#### Adapters
- **What**: Add small bottleneck layers between transformer layers
- **Benefits**: Only train the adapter layers

#### Prompt Tuning
- **What**: Learn soft prompts (continuous embeddings)
- **Benefits**: Minimal parameters to train

### 3.3 Few-Shot Learning
- **What**: Learn from very few examples
- **Techniques**: In-context learning, meta-learning

### Comparison Table:

| Method | Trainable Params | Memory Usage | Performance | Use Case |
|--------|------------------|--------------|-------------|----------|
| Full Fine-tuning | 100% | High | Best | Small-medium models |
| LoRA | 1-5% | Low | Very Good | Large models |
| Adapters | 2-8% | Medium | Good | Medium models |
| Prompt Tuning | <1% | Very Low | Good | Very large models |

## 4. Setting up the Environment {#setup}

Let's start by installing the necessary libraries and setting up our environment.

In [15]:
# Install required packages
# Run this in terminal: pip install transformers datasets torch accelerate peft evaluate

# For this notebook, we'll import the essential libraries
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    AutoModelForCausalLM,
    TrainingArguments, 
    Trainer,
    pipeline
)
from datasets import Dataset, load_dataset
from peft import LoraConfig, get_peft_model, TaskType
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

✅ All libraries imported successfully!
PyTorch version: 2.6.0
CUDA available: False


## 5. Data Preparation {#data-prep}

### 5.1 Understanding Your Data

Before fine-tuning, you need to understand:
- **Task type**: Classification, generation, question-answering, etc.
- **Data format**: Text pairs, single texts, structured data
- **Data quality**: Clean, consistent, representative
- **Data size**: How much data you have affects your approach

### 5.2 Data Requirements by Task:

| Task | Minimum Samples | Recommended | Format |
|------|----------------|-------------|--------|
| Text Classification | 100-500 per class | 1000+ per class | `{"text": "...", "label": 0}` |
| Text Generation | 1000+ | 10000+ | `{"input": "...", "output": "..."}` |
| Question Answering | 500+ | 5000+ | `{"question": "...", "context": "...", "answer": "..."}` |

### 5.3 Data Preprocessing Steps:
1. **Cleaning**: Remove noise, fix encoding issues
2. **Tokenization**: Convert text to model-readable format
3. **Splitting**: Train/validation/test splits
4. **Augmentation**: Increase data diversity (optional)

Let's implement a practical example:

In [16]:
# Example: Creating a sample sentiment analysis dataset
# In practice, you'd load your own data

# Sample data for demonstration
sample_data = {
    'text': [
        "I love this product! It's amazing.",
        "This is terrible, worst purchase ever.",
        "Pretty good, would recommend.",
        "Not worth the money, disappointed.",
        "Excellent quality and fast shipping!",
        "Okay product, nothing special.",
        "Hate it, returning immediately.",
        "Best purchase I've made this year!"
    ],
    'label': [1, 0, 1, 0, 1, 1, 0, 1]  # 1: Positive, 0: Negative
}

# Create dataset
df = pd.DataFrame(sample_data)
print("📊 Sample Dataset:")
print(df)
print(f"\n📈 Label distribution:")
print(df['label'].value_counts())

# Convert to Hugging Face dataset format
dataset = Dataset.from_pandas(df)
print(f"\n✅ Dataset created with {len(dataset)} samples")

📊 Sample Dataset:
                                     text  label
0      I love this product! It's amazing.      1
1  This is terrible, worst purchase ever.      0
2           Pretty good, would recommend.      1
3      Not worth the money, disappointed.      0
4    Excellent quality and fast shipping!      1
5          Okay product, nothing special.      1
6         Hate it, returning immediately.      0
7      Best purchase I've made this year!      1

📈 Label distribution:
label
1    5
0    3
Name: count, dtype: int64

✅ Dataset created with 8 samples


In [17]:
# Initialize tokenizer
model_name = "distilbert-base-uncased"  # Lightweight model for demonstration
tokenizer = AutoTokenizer.from_pretrained(model_name)

print(f"📝 Using tokenizer: {model_name}")
print(f"Vocabulary size: {tokenizer.vocab_size}")
print(f"Max length: {tokenizer.model_max_length}")

# Tokenization function
def tokenize_function(examples):
    """
    Tokenize the input text for the model.
    
    Args:
        examples: Dictionary with 'text' key
    
    Returns:
        Dictionary with tokenized inputs
    """
    return tokenizer(
        examples['text'], 
        padding='max_length',  # Pad to max length
        truncation=True,       # Truncate if too long
        max_length=128,        # Set reasonable max length
        return_tensors='pt'    # Return PyTorch tensors
    )

# Apply tokenization
tokenized_dataset = dataset.map(tokenize_function, batched=True)
print("\n✅ Tokenization completed!")
print(f"Dataset columns: {tokenized_dataset.column_names}")

# Example of tokenized output
print("\n🔍 Example tokenized sample:")
sample = tokenized_dataset[0]
print(f"Original text: {dataset[0]['text']}")
print(f"Token IDs: {sample['input_ids'][:10]}...")  # Show first 10 tokens
print(f"Attention mask: {sample['attention_mask'][:10]}...")  # Show first 10 values
print(f"Label: {sample['label']}")

📝 Using tokenizer: distilbert-base-uncased
Vocabulary size: 30522
Max length: 512


Map: 100%|██████████| 8/8 [00:00<00:00, 2080.25 examples/s]


✅ Tokenization completed!
Dataset columns: ['text', 'label', 'input_ids', 'attention_mask']

🔍 Example tokenized sample:
Original text: I love this product! It's amazing.
Token IDs: [101, 1045, 2293, 2023, 4031, 999, 2009, 1005, 1055, 6429]...
Attention mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]...
Label: 1


## 6. Fine-Tuning Implementation {#implementation}

### 6.1 Understanding the Training Process

Fine-tuning involves several key steps:

1. **Load Pre-trained Model**: Start with weights learned on large datasets
2. **Modify Output Layer**: Adapt for your specific task
3. **Set Training Parameters**: Learning rate, batch size, epochs
4. **Train**: Update model weights on your data
5. **Evaluate**: Test performance on validation set

### 6.2 Key Training Parameters:

- **Learning Rate**: How fast the model learns (typically 2e-5 to 5e-5 for fine-tuning)
- **Batch Size**: Number of samples processed together (8, 16, 32)
- **Epochs**: Number of times to go through the entire dataset (2-5 for fine-tuning)
- **Weight Decay**: Regularization to prevent overfitting (0.01)
- **Warmup Steps**: Gradually increase learning rate at the start

### 6.3 Training Strategies:

#### Gradual Unfreezing
```python
# Start by training only the head
for param in model.base_model.parameters():
    param.requires_grad = False

# Later, unfreeze some layers
for param in model.base_model.encoder.layer[-2:].parameters():
    param.requires_grad = True
```

#### Learning Rate Scheduling
- Start with lower learning rate for pre-trained layers
- Higher learning rate for new layers
- Gradual decay during training

Let's implement full fine-tuning:

In [18]:
# Load pre-trained model for sequence classification
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,  # Binary classification (positive/negative)
    id2label={0: "NEGATIVE", 1: "POSITIVE"},
    label2id={"NEGATIVE": 0, "POSITIVE": 1}
)

print(f"🤖 Model loaded: {model_name}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Model config: {model.config}")

# Split dataset for training and validation
train_test_split = tokenized_dataset.train_test_split(test_size=0.3, seed=42)
train_dataset = train_test_split['train']
eval_dataset = train_test_split['test']

print(f"\n📊 Dataset splits:")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(eval_dataset)}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


🤖 Model loaded: distilbert-base-uncased
Number of parameters: 66,955,010
Model config: DistilBertConfig {
  "_attn_implementation_autoset": true,
  "activation": "gelu",
  "architectures": [
    "DistilBertForMaskedLM"
  ],
  "attention_dropout": 0.1,
  "dim": 768,
  "dropout": 0.1,
  "hidden_dim": 3072,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "initializer_range": 0.02,
  "label2id": {
    "NEGATIVE": 0,
    "POSITIVE": 1
  },
  "max_position_embeddings": 512,
  "model_type": "distilbert",
  "n_heads": 12,
  "n_layers": 6,
  "pad_token_id": 0,
  "qa_dropout": 0.1,
  "seq_classif_dropout": 0.2,
  "sinusoidal_pos_embds": false,
  "tie_weights_": true,
  "torch_dtype": "float32",
  "transformers_version": "4.51.3",
  "vocab_size": 30522
}


📊 Dataset splits:
Training samples: 5
Validation samples: 3


In [19]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',              # Output directory
    num_train_epochs=3,                  # Number of training epochs
    per_device_train_batch_size=8,       # Batch size for training
    per_device_eval_batch_size=8,        # Batch size for evaluation
    warmup_steps=100,                    # Warmup steps
    weight_decay=0.01,                   # Weight decay for regularization
    learning_rate=2e-5,                  # Learning rate
    logging_dir='./logs',                # Logging directory
    logging_steps=10,                    # Log every 10 steps
    eval_strategy="epoch",         # Evaluate at the end of each epoch
    save_strategy="epoch",               # Save at the end of each epoch
    load_best_model_at_end=True,         # Load best model at the end
    metric_for_best_model="accuracy",    # Metric to use for best model
    greater_is_better=True,              # Higher accuracy is better
    report_to=None,                      # Disable wandb/tensorboard logging
    seed=42                              # For reproducibility
)

print("⚙️ Training arguments configured:")
print(f"Learning rate: {training_args.learning_rate}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Output directory: {training_args.output_dir}")

⚙️ Training arguments configured:
Learning rate: 2e-05
Batch size: 8
Epochs: 3
Output directory: ./results


In [20]:
# Define metrics for evaluation
def compute_metrics(eval_pred):
    """
    Compute metrics for evaluation.
    
    Args:
        eval_pred: EvalPrediction object with predictions and labels
    
    Returns:
        Dictionary with computed metrics
    """
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # Calculate accuracy
    accuracy = accuracy_score(labels, predictions)
    
    # Calculate precision, recall, F1
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='weighted')
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Initialize trainer
trainer = Trainer(
    model=model,                         # The model to train
    args=training_args,                  # Training arguments
    train_dataset=train_dataset,         # Training dataset
    eval_dataset=eval_dataset,           # Evaluation dataset
    compute_metrics=compute_metrics,     # Metrics function
    tokenizer=tokenizer,                 # Tokenizer for data collation
)

print("👨‍🏫 Trainer initialized successfully!")
print(f"Training dataset size: {len(trainer.train_dataset)}")
print(f"Evaluation dataset size: {len(trainer.eval_dataset)}")

👨‍🏫 Trainer initialized successfully!
Training dataset size: 5
Evaluation dataset size: 3


In [21]:
# Train the model
print("🚀 Starting training...")
print("Note: This may take a few minutes depending on your hardware\n")

# Uncomment the next line to actually train (commented to avoid long execution in demo)
# training_result = trainer.train()

# For demonstration, let's show what the training output would look like
print("📊 Training Progress (Example):")
print("""
Epoch 1/3:
Step 10: loss=0.6931, accuracy=0.5000
Step 20: loss=0.4521, accuracy=0.7500
Evaluation: accuracy=0.7000, f1=0.6800

Epoch 2/3:
Step 30: loss=0.3012, accuracy=0.8750
Step 40: loss=0.2543, accuracy=0.9000
Evaluation: accuracy=0.8500, f1=0.8400

Epoch 3/3:
Step 50: loss=0.1876, accuracy=0.9250
Step 60: loss=0.1654, accuracy=0.9500
Evaluation: accuracy=0.9000, f1=0.8900

✅ Training completed!
""")

print("💾 To save the model:")
print("trainer.save_model('./fine-tuned-model')")
print("tokenizer.save_pretrained('./fine-tuned-model')")

🚀 Starting training...
Note: This may take a few minutes depending on your hardware

📊 Training Progress (Example):

Epoch 1/3:
Step 10: loss=0.6931, accuracy=0.5000
Step 20: loss=0.4521, accuracy=0.7500
Evaluation: accuracy=0.7000, f1=0.6800

Epoch 2/3:
Step 30: loss=0.3012, accuracy=0.8750
Step 40: loss=0.2543, accuracy=0.9000
Evaluation: accuracy=0.8500, f1=0.8400

Epoch 3/3:
Step 50: loss=0.1876, accuracy=0.9250
Step 60: loss=0.1654, accuracy=0.9500
Evaluation: accuracy=0.9000, f1=0.8900

✅ Training completed!

💾 To save the model:
trainer.save_model('./fine-tuned-model')
tokenizer.save_pretrained('./fine-tuned-model')


### 6.4 Parameter-Efficient Fine-Tuning with LoRA

**LoRA (Low-Rank Adaptation)** is a technique that dramatically reduces the number of trainable parameters while maintaining performance. Instead of updating all model weights, LoRA adds small trainable matrices to existing layers.

#### How LoRA Works:
1. **Freeze** the original model weights
2. **Add** low-rank matrices (A and B) to attention layers
3. **Train** only these small matrices
4. **Merge** the adaptations with original weights for inference

#### Benefits:
- 🔥 **10-100x fewer parameters** to train
- 💾 **Reduced memory usage**
- 🚀 **Faster training**
- 📦 **Smaller model checkpoints**
- 🔄 **Easy to switch between tasks**

In [22]:
# LoRA Fine-tuning Implementation
from peft import LoraConfig, get_peft_model, TaskType

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    device_map="auto"  # Automatically handle device placement
)

print(f"📊 Base model parameters: {base_model.num_parameters():,}")

# Configure LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,  # Sequence classification task
    inference_mode=False,        # Training mode
    r=16,                       # Rank of adaptation (higher = more capacity)
    lora_alpha=32,              # LoRA scaling parameter
    lora_dropout=0.1,           # Dropout for LoRA layers
    target_modules=["q_lin", "v_lin"],  # Which modules to adapt
)

# Apply LoRA to the model
lora_model = get_peft_model(base_model, lora_config)

# Print trainable parameters
lora_model.print_trainable_parameters()

print(f"\n🎯 LoRA Configuration:")
print(f"Rank (r): {lora_config.r}")
print(f"Alpha: {lora_config.lora_alpha}")
print(f"Dropout: {lora_config.lora_dropout}")
print(f"Target modules: {lora_config.target_modules}")

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


📊 Base model parameters: 66,955,010
trainable params: 887,042 || all params: 67,842,052 || trainable%: 1.3075

🎯 LoRA Configuration:
Rank (r): 16
Alpha: 32
Dropout: 0.1
Target modules: {'v_lin', 'q_lin'}


In [23]:
# Training arguments for LoRA (can use higher learning rate)
lora_training_args = TrainingArguments(
    output_dir='./lora_results',
    num_train_epochs=5,              # Can train for more epochs with LoRA
    per_device_train_batch_size=16,  # Can use larger batch size
    per_device_eval_batch_size=16,
    learning_rate=3e-4,              # Higher learning rate for LoRA
    weight_decay=0.01,
    logging_steps=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to=None,
    seed=42
)

# Initialize LoRA trainer
lora_trainer = Trainer(
    model=lora_model,
    args=lora_training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
    tokenizer=tokenizer,
)

trainable_params, total_params = lora_model.get_nb_trainable_parameters()
print(f"🎯 LoRA Trainer configured!")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Total parameters: {total_params:,}")
print(f"Percentage trainable: {100 * trainable_params / total_params:.2f}%")

# To train: lora_trainer.train()
print("\n💡 To start LoRA training, run: lora_trainer.train()")

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


🎯 LoRA Trainer configured!
Trainable parameters: 887,042
Total parameters: 67,842,052
Percentage trainable: 1.31%

💡 To start LoRA training, run: lora_trainer.train()


## 7. Evaluation and Testing {#evaluation}

### 7.1 Evaluation Metrics

Choosing the right metrics is crucial for understanding your model's performance:

| Task | Primary Metrics | Secondary Metrics |
|------|----------------|------------------|
| **Classification** | Accuracy, F1-score | Precision, Recall, AUC |
| **Generation** | BLEU, ROUGE | Perplexity, Human evaluation |
| **QA** | Exact Match, F1 | BLEU, METEOR |
| **NER** | F1-score | Precision, Recall per entity |

### 7.2 Evaluation Best Practices:
1. **Hold-out test set**: Never seen during training
2. **Cross-validation**: For small datasets
3. **Domain-specific evaluation**: Test on real-world scenarios
4. **Error analysis**: Understand failure modes
5. **Baseline comparison**: Compare against simple baselines

In [24]:
# Comprehensive Model Evaluation
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

def evaluate_model(trainer, test_dataset, model_name="Model"):
    """
    Comprehensive model evaluation with multiple metrics and visualizations.
    
    Args:
        trainer: Trained Hugging Face trainer
        test_dataset: Test dataset
        model_name: Name for the model (for plots)
    
    Returns:
        Dictionary with evaluation results
    """
    print(f"🔍 Evaluating {model_name}...")
    
    # Get predictions
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=1)
    y_true = predictions.label_ids
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average='weighted')
    
    # Print classification report
    print(f"\n📊 {model_name} Performance:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score: {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    
    print("\n📋 Detailed Classification Report:")
    target_names = ['Negative', 'Positive']
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print("\n🎯 Confusion Matrix:")
    print(cm)
    
    return {
        'accuracy': accuracy,
        'f1': f1,
        'precision': precision,
        'recall': recall,
        'predictions': y_pred,
        'true_labels': y_true,
        'confusion_matrix': cm
    }

# Example evaluation (would run after training)
print("💡 Example evaluation output:")
print("""
🔍 Evaluating Fine-tuned Model...

📊 Fine-tuned Model Performance:
Accuracy: 0.9000
F1-Score: 0.8950
Precision: 0.9100
Recall: 0.9000

📋 Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.94      0.91        16
    Positive       0.93      0.87      0.90        15

    accuracy                           0.91        31
   macro avg       0.91      0.90      0.91        31
weighted avg       0.91      0.91      0.91        31

🎯 Confusion Matrix:
[[15  1]
 [ 2 13]]
""")

💡 Example evaluation output:

🔍 Evaluating Fine-tuned Model...

📊 Fine-tuned Model Performance:
Accuracy: 0.9000
F1-Score: 0.8950
Precision: 0.9100
Recall: 0.9000

📋 Detailed Classification Report:
              precision    recall  f1-score   support

    Negative       0.89      0.94      0.91        16
    Positive       0.93      0.87      0.90        15

    accuracy                           0.91        31
   macro avg       0.91      0.90      0.91        31
weighted avg       0.91      0.91      0.91        31

🎯 Confusion Matrix:
[[15  1]
 [ 2 13]]



In [25]:
# Model Inference and Testing
def test_model_inference(model, tokenizer, texts, model_name="Model"):
    """
    Test model inference on sample texts.
    
    Args:
        model: Trained model
        tokenizer: Model tokenizer
        texts: List of texts to test
        model_name: Name for display
    
    Returns:
        List of predictions
    """
    print(f"🧪 Testing {model_name} Inference:")
    
    # Set model to evaluation mode
    model.eval()
    
    predictions = []
    
    with torch.no_grad():
        for i, text in enumerate(texts):
            # Tokenize input
            inputs = tokenizer(
                text, 
                return_tensors="pt", 
                padding=True, 
                truncation=True, 
                max_length=128
            )
            
            # Get prediction
            outputs = model(**inputs)
            logits = outputs.logits
            predicted_class = torch.argmax(logits, dim=-1).item()
            confidence = torch.softmax(logits, dim=-1).max().item()
            
            # Convert to label
            label = "Positive" if predicted_class == 1 else "Negative"
            
            predictions.append({
                'text': text,
                'prediction': label,
                'confidence': confidence
            })
            
            print(f"Text {i+1}: {text[:50]}...")
            print(f"Prediction: {label} (Confidence: {confidence:.3f})\n")
    
    return predictions

# Test samples
test_texts = [
    "This product is absolutely amazing! I love it!",
    "Terrible quality, complete waste of money.",
    "It's okay, nothing special but does the job.",
    "Outstanding customer service and great value!",
    "Not recommended, poor performance."
]

print("🔮 Example inference results:")
for i, text in enumerate(test_texts):
    print(f"Text {i+1}: {text}")
    # Simulate prediction
    if "amazing" in text.lower() or "love" in text.lower() or "outstanding" in text.lower():
        pred, conf = "Positive", 0.95
    elif "terrible" in text.lower() or "waste" in text.lower() or "poor" in text.lower():
        pred, conf = "Negative", 0.92
    else:
        pred, conf = "Positive", 0.65
    
    print(f"Prediction: {pred} (Confidence: {conf:.3f})\n")

🔮 Example inference results:
Text 1: This product is absolutely amazing! I love it!
Prediction: Positive (Confidence: 0.950)

Text 2: Terrible quality, complete waste of money.
Prediction: Negative (Confidence: 0.920)

Text 3: It's okay, nothing special but does the job.
Prediction: Positive (Confidence: 0.650)

Text 4: Outstanding customer service and great value!
Prediction: Positive (Confidence: 0.950)

Text 5: Not recommended, poor performance.
Prediction: Negative (Confidence: 0.920)



## 8. Advanced Techniques {#advanced}

### 8.1 Learning Rate Scheduling

```python
from transformers import get_scheduler

# Linear warmup then decay
scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
```

### 8.2 Gradient Accumulation

For training with larger effective batch sizes:

```python
training_args = TrainingArguments(
    gradient_accumulation_steps=4,  # Accumulate gradients over 4 steps
    per_device_train_batch_size=8,  # Effective batch size: 8 * 4 = 32
    # ... other arguments
)
```

### 8.3 Mixed Precision Training

Reduces memory usage and speeds up training:

```python
training_args = TrainingArguments(
    fp16=True,  # Enable mixed precision
    # ... other arguments
)
```

### 8.4 Early Stopping

```python
from transformers import EarlyStoppingCallback

trainer = Trainer(
    # ... other arguments
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)
```

### 8.5 Custom Loss Functions

```python
class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get('logits')
        
        # Custom loss (e.g., focal loss for imbalanced data)
        loss_fct = FocalLoss(alpha=0.25, gamma=2.0)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss
```

In [26]:
# Advanced Training Techniques Implementation

# 1. Custom Data Collator for Dynamic Padding
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer,
    padding=True,  # Dynamic padding
    max_length=128,
    pad_to_multiple_of=8,  # Pad to multiple of 8 for efficiency
)

print("🔧 Advanced Training Components:")
print("✅ Dynamic padding data collator")

# 2. Learning Rate Finder
def find_learning_rate(trainer, start_lr=1e-7, end_lr=1e-1, num_steps=100):
    """
    Find optimal learning rate using learning rate range test.
    
    Args:
        trainer: Hugging Face trainer
        start_lr: Starting learning rate
        end_lr: Ending learning rate
        num_steps: Number of steps to test
    
    Returns:
        Recommended learning rate
    """
    print(f"🔍 Finding optimal learning rate...")
    print(f"Testing range: {start_lr} to {end_lr}")
    
    # This would implement the actual LR range test
    # For demonstration, we'll return a typical good value
    recommended_lr = 2e-5
    
    print(f"📊 Recommended learning rate: {recommended_lr}")
    return recommended_lr

# 3. Model Ensembling
class ModelEnsemble:
    """
    Ensemble multiple fine-tuned models for better performance.
    """
    def __init__(self, models, tokenizer):
        self.models = models
        self.tokenizer = tokenizer
    
    def predict(self, texts):
        """
        Get ensemble predictions from multiple models.
        """
        all_predictions = []
        
        for model in self.models:
            model.eval()
            predictions = []
            
            with torch.no_grad():
                for text in texts:
                    inputs = self.tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
                    outputs = model(**inputs)
                    logits = outputs.logits
                    probs = torch.softmax(logits, dim=-1)
                    predictions.append(probs.squeeze().numpy())
            
            all_predictions.append(predictions)
        
        # Average predictions across models
        ensemble_predictions = np.mean(all_predictions, axis=0)
        final_predictions = np.argmax(ensemble_predictions, axis=1)
        
        return final_predictions, ensemble_predictions

print("✅ Learning rate finder function")
print("✅ Model ensemble class")

# 4. Hyperparameter Optimization with Optuna
def objective(trial):
    """
    Objective function for hyperparameter optimization.
    """
    # Suggest hyperparameters
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 5e-4, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8, 16, 32])
    warmup_ratio = trial.suggest_float("warmup_ratio", 0.0, 0.2)
    weight_decay = trial.suggest_float("weight_decay", 0.0, 0.3)
    
    # Create training arguments with suggested hyperparameters
    training_args = TrainingArguments(
        output_dir='./hp_tuning',
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        warmup_ratio=warmup_ratio,
        weight_decay=weight_decay,
        num_train_epochs=2,  # Shorter for HP tuning
        evaluation_strategy="epoch",
        save_strategy="no",  # Don't save during HP tuning
        logging_steps=50,
        report_to=None,
    )
    
    # This would train and return the validation metric
    # For demo, return a simulated metric
    return 0.85  # Simulated accuracy

print("✅ Hyperparameter optimization function")
print("\n💡 To run HP optimization:")
print("import optuna")
print("study = optuna.create_study(direction='maximize')")
print("study.optimize(objective, n_trials=20)")

🔧 Advanced Training Components:
✅ Dynamic padding data collator
✅ Learning rate finder function
✅ Model ensemble class
✅ Hyperparameter optimization function

💡 To run HP optimization:
import optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)


## 9. Best Practices {#best-practices}

### 9.1 Data Best Practices

#### ✅ Do:
- **Quality over Quantity**: Clean, high-quality data is better than noisy large datasets
- **Balanced Distribution**: Ensure balanced representation across classes/categories
- **Domain Relevance**: Use data that matches your target domain
- **Validation Strategy**: Hold out 15-20% for validation, 10-15% for testing
- **Data Augmentation**: Use techniques like back-translation, paraphrasing

#### ❌ Don't:
- **Data Leakage**: Ensure no overlap between train/validation/test sets
- **Biased Sampling**: Avoid systematic biases in data collection
- **Ignore Class Imbalance**: Address severe class imbalances

### 9.2 Training Best Practices

#### ✅ Do:
- **Start Small**: Begin with a smaller model and dataset
- **Monitor Overfitting**: Watch training vs. validation metrics
- **Use Checkpointing**: Save model at different training stages
- **Learning Rate Scheduling**: Use warmup and decay
- **Gradient Clipping**: Prevent exploding gradients

#### ❌ Don't:
- **Over-train**: Stop when validation performance plateaus
- **Ignore Baselines**: Always compare against simple baselines
- **Skip Validation**: Always validate on held-out data

### 9.3 Model Selection Guidelines

| Use Case | Recommended Model | Why |
|----------|------------------|-----|
| **Text Classification** | BERT, RoBERTa | Excellent understanding |
| **Text Generation** | GPT-2, T5 | Strong generation capabilities |
| **Question Answering** | BERT, ELECTRA | Good at finding answers |
| **Multilingual** | mBERT, XLM-R | Language coverage |
| **Fast Inference** | DistilBERT, TinyBERT | Compressed models |
| **Large Scale** | Use LoRA/Adapters | Parameter efficiency |

### 9.4 Computational Efficiency

```python
# Memory optimization techniques
training_args = TrainingArguments(
    gradient_checkpointing=True,     # Trade compute for memory
    fp16=True,                       # Mixed precision
    dataloader_pin_memory=True,      # Faster data loading
    gradient_accumulation_steps=4,   # Simulate larger batch size
    per_device_train_batch_size=8,   # Adjust based on GPU memory
)
```

In [27]:
# Monitoring and Debugging Tools

import time
import psutil
import gc

class TrainingMonitor:
    """
    Monitor training progress and system resources.
    """
    def __init__(self):
        self.start_time = None
        self.step_times = []
        self.memory_usage = []
    
    def start_training(self):
        """Start monitoring training."""
        self.start_time = time.time()
        print("🚀 Training monitoring started!")
    
    def log_step(self, step, loss, lr=None):
        """Log training step information."""
        current_time = time.time()
        step_time = current_time - (self.start_time + sum(self.step_times))
        self.step_times.append(step_time)
        
        # Memory usage
        memory_mb = psutil.virtual_memory().used / 1024 / 1024
        self.memory_usage.append(memory_mb)
        
        # GPU memory (if available)
        gpu_memory = "N/A"
        if torch.cuda.is_available():
            gpu_memory = f"{torch.cuda.memory_allocated() / 1024**2:.1f}MB"
        
        print(f"Step {step}: Loss={loss:.4f}, Time={step_time:.2f}s, "
              f"RAM={memory_mb:.0f}MB, GPU={gpu_memory}")
        
        if lr:
            print(f"Learning Rate: {lr:.2e}")
    
    def get_summary(self):
        """Get training summary."""
        total_time = sum(self.step_times)
        avg_step_time = np.mean(self.step_times) if self.step_times else 0
        max_memory = max(self.memory_usage) if self.memory_usage else 0
        
        return {
            'total_time': total_time,
            'avg_step_time': avg_step_time,
            'max_memory_mb': max_memory,
            'num_steps': len(self.step_times)
        }

def debug_model_parameters(model):
    """
    Debug model parameters and gradients.
    """
    print("🔍 Model Parameter Debug:")
    
    total_params = 0
    trainable_params = 0
    frozen_params = 0
    
    for name, param in model.named_parameters():
        total_params += param.numel()
        
        if param.requires_grad:
            trainable_params += param.numel()
            # Check for gradient issues
            if param.grad is not None:
                grad_norm = param.grad.norm().item()
                if grad_norm > 100:  # Large gradient
                    print(f"⚠️  Large gradient in {name}: {grad_norm:.2f}")
                elif grad_norm < 1e-8:  # Very small gradient
                    print(f"⚠️  Tiny gradient in {name}: {grad_norm:.2e}")
        else:
            frozen_params += param.numel()
    
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,} ({100*trainable_params/total_params:.1f}%)")
    print(f"Frozen parameters: {frozen_params:,} ({100*frozen_params/total_params:.1f}%)")

def memory_cleanup():
    """
    Clean up memory.
    """
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print("🧹 Memory cleanup completed")

# Example usage
monitor = TrainingMonitor()
print("✅ Training monitor initialized")
print("✅ Debug functions ready")
print("\n💡 Usage examples:")
print("monitor.start_training()")
print("monitor.log_step(step=1, loss=0.5, lr=2e-5)")
print("debug_model_parameters(model)")
print("memory_cleanup()")

✅ Training monitor initialized
✅ Debug functions ready

💡 Usage examples:
monitor.start_training()
monitor.log_step(step=1, loss=0.5, lr=2e-5)
debug_model_parameters(model)
memory_cleanup()


## 10. Troubleshooting {#troubleshooting}

### 10.1 Common Issues and Solutions

#### 🔥 Out of Memory (OOM) Errors

**Problem**: GPU runs out of memory during training

**Solutions**:
```python
# Reduce batch size
training_args.per_device_train_batch_size = 4

# Enable gradient checkpointing
training_args.gradient_checkpointing = True

# Use gradient accumulation
training_args.gradient_accumulation_steps = 4

# Enable mixed precision
training_args.fp16 = True

# Use a smaller model
model_name = "distilbert-base-uncased"  # Instead of bert-large
```

#### 📉 Poor Performance

**Problem**: Model not learning or poor accuracy

**Diagnosis**:
```python
# Check learning rate
if loss_not_decreasing:
    # Try different learning rates: 1e-5, 2e-5, 5e-5
    
# Check data quality
if predictions_random:
    # Verify labels are correct
    # Check for data leakage
    # Ensure balanced classes
    
# Check overfitting
if train_acc_high_but_val_acc_low:
    # Add regularization
    # Reduce model complexity
    # Get more data
```

#### 🐌 Slow Training

**Problem**: Training takes too long

**Solutions**:
```python
# Optimize data loading
training_args.dataloader_num_workers = 4
training_args.dataloader_pin_memory = True

# Use larger batch size (if memory allows)
training_args.per_device_train_batch_size = 16

# Reduce sequence length
max_length = 128  # Instead of 512
```

### 10.2 Debugging Checklist

#### ✅ Before Training:
- [ ] Data is properly formatted
- [ ] Labels are correct
- [ ] Train/val split is proper
- [ ] Tokenization works correctly
- [ ] Model architecture matches task

#### ✅ During Training:
- [ ] Loss is decreasing
- [ ] Learning rate is appropriate
- [ ] No gradient explosion/vanishing
- [ ] Validation performance improves
- [ ] No memory issues

#### ✅ After Training:
- [ ] Model saves correctly
- [ ] Inference works
- [ ] Performance meets expectations
- [ ] Model generalizes to test set

### 10.3 Performance Optimization Tips

| Issue | Solution | Code Example |
|-------|----------|--------------|
| **Slow tokenization** | Use `fast=True` | `AutoTokenizer.from_pretrained(model, use_fast=True)` |
| **Memory leaks** | Clear cache | `torch.cuda.empty_cache()` |
| **Slow data loading** | Increase workers | `dataloader_num_workers=4` |
| **Large model size** | Use model compression | `model = torch.quantization.quantize_dynamic(model)` |
| **Inference latency** | Batch inference | `model(batch_inputs)` instead of loops |

In [28]:
# Complete Fine-tuning Pipeline Summary

def complete_finetuning_pipeline(
    model_name="distilbert-base-uncased",
    train_texts=None,
    train_labels=None,
    test_texts=None,
    use_lora=True,
    num_epochs=3
):
    """
    Complete fine-tuning pipeline from start to finish.
    
    This function demonstrates the entire process:
    1. Data preparation
    2. Model loading
    3. Training configuration
    4. Training
    5. Evaluation
    6. Inference
    
    Args:
        model_name: Pretrained model to use
        train_texts: Training texts
        train_labels: Training labels
        test_texts: Test texts for inference
        use_lora: Whether to use LoRA
        num_epochs: Number of training epochs
    
    Returns:
        Trained model and results
    """
    
    print("🚀 Starting Complete Fine-tuning Pipeline")
    print("=" * 50)
    
    # Step 1: Data Preparation
    print("📊 Step 1: Data Preparation")
    if train_texts is None:
        train_texts = sample_data['text']
        train_labels = sample_data['label']
    
    # Step 2: Tokenization
    print("🔤 Step 2: Tokenization")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Step 3: Model Loading
    print("🤖 Step 3: Model Loading")
    if use_lora:
        print("Using LoRA for parameter-efficient fine-tuning")
        # LoRA setup would go here
    else:
        print("Using full fine-tuning")
        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=2
        )
    
    # Step 4: Training Configuration
    print("⚙️ Step 4: Training Configuration")
    training_args = TrainingArguments(
        output_dir='./results',
        num_train_epochs=num_epochs,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        report_to=None,
    )
    
    # Step 5: Training (simulated)
    print("🏋️ Step 5: Training (Simulated)")
    print(f"Training for {num_epochs} epochs...")
    
    # Step 6: Evaluation (simulated)
    print("📈 Step 6: Evaluation")
    results = {
        'accuracy': 0.89,
        'f1': 0.88,
        'precision': 0.90,
        'recall': 0.87
    }
    
    print(f"Final Results: {results}")
    
    # Step 7: Inference Test
    print("🔮 Step 7: Inference Test")
    if test_texts is None:
        test_texts = [
            "This is amazing!",
            "I hate this product.",
            "Pretty good overall."
        ]
    
    for text in test_texts:
        # Simulated prediction
        prediction = "Positive" if any(word in text.lower() for word in ['amazing', 'good', 'love']) else "Negative"
        confidence = 0.92 if prediction == "Positive" else 0.88
        print(f"Text: '{text}' → {prediction} ({confidence:.2f})")
    
    print("\n✅ Pipeline completed successfully!")
    return results

# Run the complete pipeline
print("🎯 Running Complete Fine-tuning Pipeline Demo:")
results = complete_finetuning_pipeline(
    model_name="distilbert-base-uncased",
    use_lora=True,
    num_epochs=3
)

print("\n" + "="*60)
print("🎊 CONGRATULATIONS! 🎊")
print("You've completed the Fine-tuning with Transformers guide!")
print("\n📚 What you've learned:")
print("✅ Fine-tuning fundamentals")
print("✅ Data preparation techniques")
print("✅ Full fine-tuning implementation")
print("✅ Parameter-efficient methods (LoRA)")
print("✅ Evaluation and testing")
print("✅ Advanced techniques")
print("✅ Best practices and troubleshooting")
print("\n🚀 Next steps:")
print("1. Try with your own dataset")
print("2. Experiment with different models")
print("3. Explore other PEFT methods")
print("4. Deploy your fine-tuned model")
print("="*60)

🎯 Running Complete Fine-tuning Pipeline Demo:
🚀 Starting Complete Fine-tuning Pipeline
📊 Step 1: Data Preparation
🔤 Step 2: Tokenization
🤖 Step 3: Model Loading
Using LoRA for parameter-efficient fine-tuning
⚙️ Step 4: Training Configuration
🏋️ Step 5: Training (Simulated)
Training for 3 epochs...
📈 Step 6: Evaluation
Final Results: {'accuracy': 0.89, 'f1': 0.88, 'precision': 0.9, 'recall': 0.87}
🔮 Step 7: Inference Test
Text: 'This is amazing!' → Positive (0.92)
Text: 'I hate this product.' → Negative (0.88)
Text: 'Pretty good overall.' → Positive (0.92)

✅ Pipeline completed successfully!

🎊 CONGRATULATIONS! 🎊
You've completed the Fine-tuning with Transformers guide!

📚 What you've learned:
✅ Fine-tuning fundamentals
✅ Data preparation techniques
✅ Full fine-tuning implementation
✅ Parameter-efficient methods (LoRA)
✅ Evaluation and testing
✅ Advanced techniques
✅ Best practices and troubleshooting

🚀 Next steps:
1. Try with your own dataset
2. Experiment with different models
3. Exp

## 📚 Additional Resources and References

### 🔗 Essential Libraries
- **[Transformers](https://huggingface.co/docs/transformers/)**: Main library for transformer models
- **[Datasets](https://huggingface.co/docs/datasets/)**: Easy dataset loading and processing
- **[PEFT](https://github.com/huggingface/peft)**: Parameter-Efficient Fine-Tuning methods
- **[Accelerate](https://huggingface.co/docs/accelerate/)**: Distributed training made easy

### 📖 Learning Resources
- **[Hugging Face Course](https://huggingface.co/learn/nlp-course/)**: Comprehensive NLP course
- **[Fine-tuning Guide](https://huggingface.co/docs/transformers/training)**: Official fine-tuning documentation
- **[PEFT Paper](https://arxiv.org/abs/2106.09685)**: LoRA research paper

### 🛠 Practical Tools
- **[Model Hub](https://huggingface.co/models)**: Pre-trained models
- **[Spaces](https://huggingface.co/spaces)**: Deploy and share your models
- **[Gradio](https://gradio.app/)**: Create web interfaces for models

### 💡 Advanced Topics to Explore
1. **Multi-task Learning**: Train on multiple tasks simultaneously
2. **Domain Adaptation**: Adapt models to specific domains
3. **Continual Learning**: Learn new tasks without forgetting old ones
4. **Model Compression**: Reduce model size for deployment
5. **Federated Learning**: Train models across distributed data

---

## 🎯 Quick Reference Commands

```bash
# Install dependencies
pip install transformers datasets torch accelerate peft evaluate

# Download model
from transformers import AutoModel
model = AutoModel.from_pretrained("bert-base-uncased")

# Fine-tune with Trainer
trainer = Trainer(model=model, args=training_args, train_dataset=train_data)
trainer.train()

# Save model
trainer.save_model("./my-fine-tuned-model")

# Load and use
model = AutoModel.from_pretrained("./my-fine-tuned-model")
```

---

**Happy Fine-tuning! 🚀**

*This notebook provides a comprehensive introduction to fine-tuning with Transformers. Remember to experiment with different approaches and datasets to find what works best for your specific use case.*